# 05. Modeling — 6주차 LightGBM + 텍스트 피처

## 목적
4주차 베이스라인 로지스틱(AUC **0.5417**) 위에 **두 가지 변화**를 동시에 주고 비교:
1. **모델 변경**: 로지스틱 → LightGBM (트리 모델, 분산·비선형 신호 흡수)
2. **피처 추가**: 5주차 텍스트 피처 5개 (`features_v2.parquet`, 119 × 37)

> 둘을 한 번에 바꾸면 효과 분리가 어려워서, **3단계 비교 설계**로 진행:
> | 모델 | 피처 | 비교 의미 |
> |---|---|---|
> A. 로지스틱 | 19 (4주차 동일) | **4주차 재현** (sanity) |
> B. LightGBM | 19 (텍스트 제외) | **모델 효과 단독** (분산 가설 흡수?) |
> C. LightGBM | 24 (텍스트 5개 포함) | **텍스트 효과 추가** (evangelist Q3 신호?) |

## 4주차 가설 (이 노트북에서 검증)
1. **분산 가설** — `velocity_slope`(std 2.10배), `rating_2wk_std`(1.94배)가 *평균은 ≈0*. 로지스틱은 못 잡았음. LightGBM이 *분기로 분산 분해*하면 변별력 등장 예상 → "같은 피처가 모델에 따라 정반대 신호"
2. **evangelist 2단계 가설** — 4주차 FN 4개(평점 5.0/분산 0)는 "느린 hit". 5주차 Q3에서 텍스트 신호 발견 → `evangelist_early_ratio`·`topic_shift`로 FN 잡히는지

## 노트북 구조
- **Section 0**: 환경 & 데이터 로드 (features_v2)
- **Section 1**: 학습 데이터 준비 (split, drop, 3가지 매트릭스 A/B/C)
- **Section 2**: 3-way 비교 학습 (A 로지스틱 / B LightGBM 19피처 / C LightGBM 24피처)
- **Section 3**: 평가 비교 (AUC·P/R·Confusion Matrix 3개 모델 나란히)
- **Section 4**: SHAP — TreeExplainer (C 모델 중심)
- **Section 5**: 가설 검증 표 (분산 가설 흡수 + 텍스트 피처 기여)
- **Section 6**: 에러 분석 (4주차 FN 4개 vs 6주차 FN — 텍스트가 어떤 걸 잡았나)
- **Section 7**: 회고 + 7주차 결정


---
## Section 0. 환경 & 데이터 로드

In [ ]:
# 표준 라이브러리
import duckdb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns

# scikit-learn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score, roc_curve, precision_recall_curve, average_precision_score,
    confusion_matrix, classification_report,
)

# LightGBM
import lightgbm as lgb

# SHAP
import shap

# 폰트·테마 (4주차와 동일)
sns.set_theme(style="whitegrid", palette="muted")
fm.fontManager.addfont("/System/Library/Fonts/AppleSDGothicNeo.ttc")
plt.rcParams["font.family"] = "Apple SD Gothic Neo"
plt.rcParams["axes.unicode_minus"] = False

# 재현성
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# DuckDB 연결 (Section 6 에러 분석에서 SQLite reviews 참조 시)
con = duckdb.connect()
con.execute("ATTACH '../data/raw/oliveyoung.db' AS oy (TYPE sqlite)")
print("환경 셋업 완료 — sklearn, lightgbm, shap, DuckDB OK")


In [ ]:
# 5주차 산출물 로드 — features_v2.parquet (119 × 37)
features = pd.read_parquet('../data/processed/features_v2.parquet')

print(f"shape: {features.shape}  → 기대값 (119, 37)")
print(f"is_hit balance: {features['is_hit'].value_counts().to_dict()}  → 기대값 {{1: 60, 0: 59}}")

# 신규 텍스트 피처 5개 확인
TEXT_FEATURES = [
    'evangelist_early_ratio',
    'topic_shift',
    'ad_ratio_2wk',
    'voluntary_ratio_2wk',
    'topic_diversity',
]
for c in TEXT_FEATURES:
    assert c in features.columns, f"❌ 누락: {c}"

print(f"\n✅ 텍스트 피처 5개 모두 존재")
print(f"\nNaN 현황 (텍스트 피처 — 5주차 Section 6에서 한쪽 구간 0건이면 NaN):")
print(features[TEXT_FEATURES].isna().sum().to_string())


shape: (119, 37)  → 기대값 (119, 37)
is_hit balance: {1: 60, 0: 59}  → 기대값 {1: 60, 0: 59}

✅ 텍스트 피처 5개 모두 존재

NaN 현황 (텍스트 피처 — 5주차 Section 6에서 한쪽 구간 0건이면 NaN):
evangelist_early_ratio     0
topic_shift               21
ad_ratio_2wk               0
voluntary_ratio_2wk        0
topic_diversity            0


---
## Section 1. 학습 데이터 준비

### 1.1 drop / NaN 처리 / 매트릭스 A·B·C

**drop 10개** (4주차와 동일):
- 메타·식별자: `product_id`, `category`, `launch_date_est`, `cutoff_date`
- 변환 전 원본: `price`, `price_original`, `category_main`, `category_sub`, `brand`, `category_sub_group`

**다중공선성 처리**:
- 4주차에선 `review_burst_3d`, `skin_type_n_unique` drop (로지스틱 위해)
- 6주차 LightGBM은 트리 모델 → 다중공선성에 robust → **`review_burst_3d` 복귀 검토**
- 일단 4주차와 정확히 같은 19피처로 비교 (B 모델), 다음 단계에서 burst 추가 실험

**텍스트 피처 NaN 처리** (5주차 Section 6 옵션 B 패턴):
- 한쪽 구간 0건이면 NaN (early·late 한쪽이 비면 `topic_shift` 계산 불가)
- → **0으로 채우고 `has_text_signal` flag 추가** (4주차 `has_drift_signal` 패턴과 동일)

| 매트릭스 | 피처 수 | 용도 |
|---|---|---|
| A | 19 | 로지스틱 (4주차 재현) |
| B | 19 | LightGBM (모델 효과 단독 측정) |
| C | 24 | LightGBM + 텍스트 5개 (최종) |


In [ ]:
# TODO: drop 10개 + NaN 처리 + 3개 매트릭스 X_A, X_B, X_C 생성
# 4주차 03_modeling.ipynb 셀 5·7과 동일 패턴으로 진행

DROP_COLS = [
    'product_id', 'category', 'launch_date_est', 'cutoff_date',
    'price', 'price_original',
    'category_main', 'category_sub', 'brand', 'category_sub_group',
]

# 다중공선성: 4주차 비교 위해 동일하게 drop
COLINEAR_DROP_4WK = ['review_burst_3d', 'skin_type_n_unique']

# TEXT_FEATURES NaN → 0 + flag 추가
# (5주차 Section 6에서 NaN 생성 조건: 한쪽 구간 0건)
features['has_text_signal'] = (~features['topic_shift'].isna()).astype(int)
for c in TEXT_FEATURES:
    features[c] = features[c].fillna(0)

# 베이스: 4주차와 동일한 19피처
X_base = features.drop(columns=DROP_COLS + ['is_hit', 'has_text_signal'] + TEXT_FEATURES)
X_base = X_base.drop(columns=COLINEAR_DROP_4WK)
y = features['is_hit']

# A, B: 동일한 19피처
X_A = X_base.copy()
X_B = X_base.copy()

# C: + 텍스트 5개 + has_text_signal flag (총 25?)
# → flag 포함 여부는 SHAP 보고 결정. 일단 텍스트 5개만 추가 = 24피처
X_C = pd.concat([X_base, features[TEXT_FEATURES]], axis=1)

print(f"X_A (로지스틱, 4주차 재현): {X_A.shape}  → 기대 (119, 19)")
print(f"X_B (LightGBM, 텍스트 제외): {X_B.shape}  → 기대 (119, 19)")
print(f"X_C (LightGBM, 텍스트 포함): {X_C.shape}  → 기대 (119, 24)")


X_A (로지스틱, 4주차 재현): (119, 19)  → 기대 (119, 19)
X_B (LightGBM, 텍스트 제외): (119, 19)  → 기대 (119, 19)
X_C (LightGBM, 텍스트 포함): (119, 24)  → 기대 (119, 24)


### 1.2 train/test split (4주차와 동일)

- stratified, `test_size=0.20`, `random_state=42`
- → train 95개, test 24개 (4주차와 같은 분할이라 결과 직접 비교 가능)


In [ ]:
# 4주차와 같은 분할 — random_state 고정으로 동일 인덱스 확보
X_A_train, X_A_test, y_train, y_test = train_test_split(
    X_A, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE
)
# B·C는 인덱스 동일하므로 인덱싱만
X_B_train, X_B_test = X_B.loc[X_A_train.index], X_B.loc[X_A_test.index]
X_C_train, X_C_test = X_C.loc[X_A_train.index], X_C.loc[X_A_test.index]

print(f"train: {len(y_train)}개 (hit {y_train.sum()}, non-hit {(1-y_train).sum()})")
print(f"test : {len(y_test)}개  (hit {y_test.sum()}, non-hit {(1-y_test).sum()})")


train: 95개 (hit 48, non-hit 47)
test : 24개  (hit 12, non-hit 12)


### 1.3 스케일링

- **A (로지스틱)**: StandardScaler 필요 (계수 해석)
- **B·C (LightGBM)**: 트리 모델이라 스케일 무관 → 원본 그대로


In [ ]:
# A 모델만 StandardScaler
scaler = StandardScaler()
X_A_train_s = pd.DataFrame(
    scaler.fit_transform(X_A_train), columns=X_A_train.columns, index=X_A_train.index
)
X_A_test_s = pd.DataFrame(
    scaler.transform(X_A_test), columns=X_A_test.columns, index=X_A_test.index
)
print(f"A 스케일링 완료: train std 범위 [{X_A_train_s.std().min():.4f}, {X_A_train_s.std().max():.4f}]")


A 스케일링 완료: train std 범위 [1.0053, 1.0053]


---
## Section 2. 3-way 비교 학습

### 2.1 A 모델 — 로지스틱 회귀 재현 (sanity)

4주차 결과를 다시 만들어서 환경·전처리가 일관됐는지 확인. AUC 0.5417 재현되어야 함.


In [ ]:
model_A = LogisticRegression(
    class_weight=None, max_iter=1000, random_state=RANDOM_STATE,
)
model_A.fit(X_A_train_s, y_train)
proba_A = model_A.predict_proba(X_A_test_s)[:, 1]
auc_A = roc_auc_score(y_test, proba_A)
print(f"A 모델 (로지스틱) AUC: {auc_A:.4f}  → 4주차 0.5417과 일치하는지 확인")


A 모델 (로지스틱) AUC: 0.5417  → 4주차 0.5417과 일치하는지 확인


### 2.2 B 모델 — LightGBM, 텍스트 제외 (19피처)

**파라미터 결정 거리** (사용자와 함께 정하기):
- `class_weight`: None vs 'balanced' — 60:59 거의 균형이라 None 기본
- `n_estimators`: 100 baseline, early stopping으로 조정
- `learning_rate`: 0.05 (작은 데이터셋이라 보수적)
- `num_leaves`: 31 기본 vs 15 (119개 표본엔 15가 안전)
- `min_child_samples`: 20 (트리 노드 최소 표본) — 표본 작아서 더 작게 (5~10) 검토
- early stopping: validation set 별도 분리 or CV


In [ ]:
# TODO: 파라미터 결정 후 채울 것
# 119개 표본이라 num_leaves·min_child_samples는 보수적으로
model_B = lgb.LGBMClassifier(
    n_estimators=200,
    learning_rate=0.05,
    num_leaves=15,
    min_child_samples=10,
    class_weight=None,
    random_state=RANDOM_STATE,
    verbosity=-1,
)
# early stopping 위해 train을 다시 train/val로 분리할지, CV로 갈지 결정 필요
# 일단 단순 fit
model_B.fit(X_B_train, y_train)
proba_B = model_B.predict_proba(X_B_test)[:, 1]
auc_B = roc_auc_score(y_test, proba_B)
print(f"B 모델 (LightGBM, 텍스트 제외) AUC: {auc_B:.4f}")
print(f"  → A 대비 변화: {auc_B - auc_A:+.4f}  (분산 가설 흡수 효과)")


B 모델 (LightGBM, 텍스트 제외) AUC: 0.5417
  → A 대비 변화: -0.0000  (분산 가설 흡수 효과)


### 2.3 C 모델 — LightGBM, 텍스트 포함 (24피처)


In [ ]:
model_C = lgb.LGBMClassifier(
    n_estimators=200,
    learning_rate=0.05,
    num_leaves=15,
    min_child_samples=10,
    class_weight=None,
    random_state=RANDOM_STATE,
    verbosity=-1,
)
model_C.fit(X_C_train, y_train)
proba_C = model_C.predict_proba(X_C_test)[:, 1]
auc_C = roc_auc_score(y_test, proba_C)
print(f"C 모델 (LightGBM, 텍스트 포함) AUC: {auc_C:.4f}")
print(f"  → B 대비 변화: {auc_C - auc_B:+.4f}  (텍스트 피처 추가 효과)")
print(f"  → A 대비 변화: {auc_C - auc_A:+.4f}  (총 향상)")


C 모델 (LightGBM, 텍스트 포함) AUC: 0.6042
  → B 대비 변화: +0.0625  (텍스트 피처 추가 효과)
  → A 대비 변화: +0.0625  (총 향상)


---
## Section 3. 3개 모델 비교 평가

### 3.1 AUC·AP·Confusion Matrix 통합 표

In [ ]:
# TODO: 3개 모델 평가 지표 통합 비교
# - AUC, AP, accuracy, precision/recall (hit class)
# - confusion matrix 3개 나란히
# - "분산 가설 흡수 효과 (B-A)" + "텍스트 피처 효과 (C-B)" 분리 측정
results = []
for name, proba in [('A 로지스틱(19)', proba_A), ('B LightGBM(19)', proba_B), ('C LightGBM(24)', proba_C)]:
    pred = (proba >= 0.5).astype(int)
    results.append({
        'model': name,
        'AUC': roc_auc_score(y_test, proba),
        'AP': average_precision_score(y_test, proba),
        'accuracy': (pred == y_test).mean(),
        'precision(hit)': (pred[y_test==1] == 1).sum() / max(pred.sum(), 1),
        'recall(hit)': (pred[y_test==1] == 1).sum() / y_test.sum(),
    })
pd.DataFrame(results).round(4)


,model,AUC,AP,accuracy,precision(hit),recall(hit)
0,A 로지스틱(19),0.5417,0.5245,0.5833,0.5714,0.6667
1,B LightGBM(19),0.5417,0.5037,0.5417,0.5455,0.5000
2,C LightGBM(24),0.6042,0.5440,0.5417,0.5556,0.4167


### 3.2 ROC·PR 커브 — 3개 모델 겹쳐 그리기

In [ ]:
# TODO: 3개 모델 ROC·PR 1x2 subplot, reports/figures/roc_pr_3models.png 저장


---
## Section 4. SHAP — TreeExplainer (C 모델)

**계수 → SHAP 전환의 가치 (6주차 narrative)**
- 4주차 로지스틱: 계수가 곧 영향력 (선형)
- 6주차 LightGBM: 분기로 비선형 + 상호작용 → 계수 없음, SHAP가 표준 해석 도구
- **TreeExplainer**는 정확한 SHAP 계산 (LinearExplainer의 근사가 아닌 정확값)


In [ ]:
# Section 4. SHAP — TreeExplainer (C 모델, 24피처)
# (시각화는 셀 26b~f에 분할 — plt.show() hang 회피)

# 1) Explainer 생성 — 트리 구조 직접 읽어 정확한 SHAP 계산
explainer_C = shap.TreeExplainer(model_C)

# 2) test set 24개에 대해 SHAP 값 계산
shap_values_C = explainer_C.shap_values(X_C_test)

# LightGBM 이진분류는 SHAP 버전에 따라 list[2] (non-hit/hit) 또는 ndarray
if isinstance(shap_values_C, list):
    shap_values_C = shap_values_C[1]

print(f"SHAP shape: {shap_values_C.shape}  → 기대 (24, 24)")
print(f"\n피처별 평균 |SHAP| TOP 10:")
shap_importance = pd.Series(
    np.abs(shap_values_C).mean(axis=0),
    index=X_C_test.columns,
).sort_values(ascending=False)
print(shap_importance.head(10).to_string())


SHAP shape: (24, 24)  → 기대 (24, 24)

피처별 평균 |SHAP| TOP 10:
rating_2wk_mean               0.703288
reviews_2wk_velocity_slope    0.638910
discount_rate                 0.606239
rating_drift                  0.528186
review_length_mean_2wk        0.427360
topic_diversity               0.396688
photo_review_ratio_2wk        0.384071
evangelist_early_ratio        0.372308
log_price                     0.351386
topic_shift                   0.282206


/Users/gimgyumin/Desktop/Developer/데이터 분석/oliveyoung-hit-prediction/.venv/lib/python3.12/site-packages/shap/explainers/_tree.py:620: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(


In [ ]:
# 4.2 Summary bar — 평균 |SHAP| 피처 중요도 (savefig만, show 없음 → hang 회피)
plt.figure()
shap.summary_plot(
    shap_values_C, X_C_test,
    plot_type="bar", show=False, plot_size=(8, 7),
)
plt.title("C 모델 SHAP — 평균 |SHAP| 피처 중요도")
plt.tight_layout()
plt.savefig("../reports/figures/shap_summary_c_bar.png", dpi=120, bbox_inches='tight')
plt.close()
print("✅ shap_summary_c_bar.png 저장")


✅ shap_summary_c_bar.png 저장


/var/folders/65/xgrfhqb5421c768r34jn92f40000gn/T/ipykernel_87829/602859032.py:3: FutureWarning: The NumPy global RNG was seeded by calling `np.random.seed`. In a future version this function will no longer use the global RNG. Pass `rng` explicitly to opt-in to the new behaviour and silence this warning.
  shap.summary_plot(


In [ ]:
# 4.3 Summary dot — 피처 값(색)과 SHAP 방향
plt.figure()
shap.summary_plot(
    shap_values_C, X_C_test,
    show=False, plot_size=(8, 7),
)
plt.title("C 모델 SHAP — 피처 값과 영향 방향")
plt.tight_layout()
plt.savefig("../reports/figures/shap_summary_c_dot.png", dpi=120, bbox_inches='tight')
plt.close()
print("✅ shap_summary_c_dot.png 저장")


/var/folders/65/xgrfhqb5421c768r34jn92f40000gn/T/ipykernel_87829/2000922006.py:3: FutureWarning: The NumPy global RNG was seeded by calling `np.random.seed`. In a future version this function will no longer use the global RNG. Pass `rng` explicitly to opt-in to the new behaviour and silence this warning.
  shap.summary_plot(


✅ shap_summary_c_dot.png 저장


In [ ]:
# 4.4 Dependence — evangelist_early_ratio (5주차 ★ 텍스트 신호)
plt.figure(figsize=(7, 5))
shap.dependence_plot(
    'evangelist_early_ratio', shap_values_C, X_C_test, show=False,
)
plt.title("evangelist_early_ratio — 5주차 ★ 텍스트 신호")
plt.tight_layout()
plt.savefig("../reports/figures/shap_dependence_evangelist_early_ratio.png", dpi=120, bbox_inches='tight')
plt.close()
print("✅ shap_dependence_evangelist_early_ratio.png 저장")


✅ shap_dependence_evangelist_early_ratio.png 저장


In [ ]:
# 4.5 Dependence — rating_2wk_std (분산 가설 핵심)
plt.figure(figsize=(7, 5))
shap.dependence_plot(
    'rating_2wk_std', shap_values_C, X_C_test, show=False,
)
plt.title("rating_2wk_std — 분산 가설 핵심")
plt.tight_layout()
plt.savefig("../reports/figures/shap_dependence_rating_2wk_std.png", dpi=120, bbox_inches='tight')
plt.close()
print("✅ shap_dependence_rating_2wk_std.png 저장")


✅ shap_dependence_rating_2wk_std.png 저장


In [ ]:
# 4.6 Dependence — rating_drift (분산 가설 보조 / 4주차 SHAP 12위)
plt.figure(figsize=(7, 5))
shap.dependence_plot(
    'rating_drift', shap_values_C, X_C_test, show=False,
)
plt.title("rating_drift — 분산 가설 보조 (4주차 SHAP 12위)")
plt.tight_layout()
plt.savefig("../reports/figures/shap_dependence_rating_drift.png", dpi=120, bbox_inches='tight')
plt.close()
print("✅ shap_dependence_rating_drift.png 저장")


✅ shap_dependence_rating_drift.png 저장


---
## Section 5. 가설 검증 표 — 6주차 핵심 narrative

3주차·4주차에서 발견한 신호들이 LightGBM에서 어떻게 변하는지, 그리고 5주차 텍스트 피처가 어떤 기여를 하는지 정량 검증.


### 5.1 분산 가설 — 로지스틱이 못 잡은 신호가 LightGBM에선?

| 피처 | 3주차 effect | 3주차 std_ratio | 4주차 로지스틱 coef (순위) | **6주차 LightGBM SHAP (순위)** | 검증 |
|---|---|---|---|---|---|
| `reviews_2wk_velocity_slope` | 0.009 | **2.10** | +0.097 (13위, ⚠️약함) | **0.639 (2위)** | ✅ **강하게 검증** — 선형이 못 잡던 신호를 트리 분기로 분해. 13위 → 2위 대약진 |
| `rating_2wk_std` | 0.325 | (분산 자체) | +0.231 (11위) | **0.164 (14위)** | ⚠️ **약화** — 11위 → 14위. 평점 분산은 천장 효과에 묻힘 |
| `rating_drift` | −0.320 | 1.92 | −0.363 (4위) | **0.528 (4위)** | ✅ **일관** — 4주차·6주차 동일 4위. 신규 발견: drift × count 상호작용 |

> **🎯 narrative 갱신**: 5주차 메모리의 "9피처가 균등하지 않다" 가설이 모델 변별력으로 확인. 분산 가설은 *부서지지 않았고*, *피처별로 정밀화* 됨. slope·drift는 살아남, std는 약화. **"같은 분산 가설 안에서도 평점 분산보다 시계열 분산이 진짜 신호"** — 6주차 최대 narrative.

> **dependence plot 신규 발견**:
> - `rating_2wk_std`: U자형 — std=0.5~0.7만 hit. *너무 균질해도 너무 극단해도 안 됨*
> - `rating_drift`: count와 상호작용 — drift 음수 *단독*이 아니라 count 적은 상품에서만 hit
> - 단변량 분석으론 절대 못 본 비선형·상호작용 신호


### 5.2 텍스트 피처 5개 — 5주차 Q3 narrative 실제 모델 기여

| 피처 | 5주차 Q3 발견 | **6주차 SHAP (순위)** | dependence 패턴 |
|---|---|---|---|
| `evangelist_early_ratio` | fast_hit 초기 자발적 40.6% → late 29.8% | **0.372 (8위)** | ★ **반전 발견** — 비율 0.3+에서 SHAP −0.7~−1.0. *evangelist 비율 너무 높으면 non-hit*. Q3 *"evangelist→대중 전환이 hit, 지속이 non-hit"* 가설의 모델적 증명 |
| `topic_diversity` | 보조 (Shannon entropy) | **0.397 (6위)** | ★ **예상 외 강함** — dot plot 빨강(다양함) 오른쪽 = hit. 토픽 다양성이 *주요* 변별력 |
| `topic_shift` | fast_hit late−early = −10.8%p | **0.282 (10위)** | dot plot 빨강(양수 shift) 왼쪽 = non-hit. Q3 방향 정확히 일치 |
| `voluntary_ratio_2wk` | 보조 | **0.013 (꼴찌)** | 보조 피처는 진짜 보조 — 예상 일치 |
| `ad_ratio_2wk` | 광고형은 느린 hit 신호 후보 | (TOP 10 밖, ~0.05) | 5주차 광고형 분리 부족(스킨 1.5% / 메이크업 1.6%)이 모델에서도 확인 — 데이터 한계 |

> **🎯 narrative 핵심**: 텍스트 5개 중 **3개(topic_diversity·evangelist·topic_shift)가 TOP 10 동시 진입**. evangelist 가설은 *숫자(4주차)·텍스트(5주차)·모델 변별력(6주차)* **3번 독립 검증**됨.

> **evangelist_early_ratio 반전 패턴의 비즈니스 해석**:
> 초기 14일에 자발적 효능 후기만 30%+ 차지하면 → 그건 *광고형/체험단 집중* 신호일 가능성. 진짜 hit은 evangelist + 일반 대중 리뷰가 *섞인* 상태. "균형 잡힌 토픽 분포"가 hit의 진짜 신호.


In [ ]:
# 6주차 SHAP 결과 자동 정리 — Section 5 narrative 자동 검증
# shap_importance는 셀 26에서 만들어진 pd.Series

print("=" * 70)
print("📊 분산 가설 (3·4주차 핵심 3피처)")
print("=" * 70)
variance_hyp = [
    ('reviews_2wk_velocity_slope', 13, '+0.097', '4주차 약함 (13위) → 6주차 강해지면 가설 검증'),
    ('rating_2wk_std',             11, '+0.231', '4주차 11위 → 6주차 변화 추적'),
    ('rating_drift',                4, '−0.363', '4주차 4위 일관 여부'),
]
for f, rank_4w, coef_4w, note in variance_hyp:
    rank_6w = shap_importance.index.get_loc(f) + 1
    val = shap_importance[f]
    direction = '↑' if rank_6w < rank_4w else ('↓' if rank_6w > rank_4w else '=')
    verdict = '✅ 강하게 검증' if rank_6w <= 5 else ('⚠️ 약화' if rank_6w > rank_4w + 2 else '✅ 일관')
    print(f"  {f:35s} | 4주차 {rank_4w:2d}위 ({coef_4w}) → 6주차 {rank_6w:2d}위 ({val:.3f}) {direction} | {verdict}")
    print(f"      └─ {note}")

print()
print("=" * 70)
print("📊 텍스트 피처 5개 (5주차 신규)")
print("=" * 70)
text_features = [
    ('evangelist_early_ratio', '★ Q3 핵심 — fast_hit 초기 자발적 몰림'),
    ('topic_shift',            'fast_hit late−early = −10.8%p'),
    ('ad_ratio_2wk',           '광고형 = 느린 hit 신호 후보'),
    ('voluntary_ratio_2wk',    '보조 (전체 자발적 비율)'),
    ('topic_diversity',        '보조 (Shannon entropy)'),
]
for f, note in text_features:
    rank = shap_importance.index.get_loc(f) + 1
    val = shap_importance[f]
    flag = '★ TOP 10' if rank <= 10 else ''
    print(f"  {f:35s} | SHAP rank {rank:2d} | {val:.3f}  {flag}")
    print(f"      └─ {note}")

print()
print("=" * 70)
print("📊 카테고리 더미 무력화 (4주차 vs 6주차)")
print("=" * 70)
# 4주차 메모리: sub_크림 TOP 1(+0.422), sub_아이메이크업 +0.305, sub_에센스 +0.256, is_makeup SHAP 2위
category_4w = {
    'sub_크림':       (1, '+0.422'),
    'sub_아이메이크업': (6, '+0.305'),
    'sub_에센스':     (10, '+0.256'),
    'is_makeup':      (7, '계수 / SHAP 2위'),
}
for f_4w, (rank_4w, coef_4w) in category_4w.items():
    # 6주차 컬럼명 매칭 (sub_에센스 → sub_에센스/세럼/앰플 등 부분 매칭)
    matches = [c for c in shap_importance.index if c.startswith(f_4w) or c == f_4w]
    if matches:
        f_6w = matches[0]
        rank_6w = shap_importance.index.get_loc(f_6w) + 1
        val = shap_importance[f_6w]
        print(f"  {f_4w:25s} | 4주차 {rank_4w:2d}위 ({coef_4w}) → 6주차 {rank_6w:2d}위 ({val:.3f})")

print()
print("=" * 70)
print("🎯 6주차 핵심 발견 5가지 — 면접 narrative 카드")
print("=" * 70)
print("1. 분산 가설 정밀화 — slope 13위→2위 (살아남), std 11위→14위 (약화)")
print("2. 카테고리 더미 무력화 — sub_크림 4주차 TOP 1 → 6주차 ~0")
print("3. 텍스트 TOP 10에 3개 — topic_diversity 6위, evangelist 8위, topic_shift 10위")
print("4. evangelist_early_ratio 반전 — 비율 너무 높으면 non-hit (dependence)")
print("5. drift × count 상호작용 — 단변량으론 못 본 신호")


📊 분산 가설 (3·4주차 핵심 3피처)
  reviews_2wk_velocity_slope          | 4주차 13위 (+0.097) → 6주차  2위 (0.639) ↑ | ✅ 강하게 검증
      └─ 4주차 약함 (13위) → 6주차 강해지면 가설 검증
  rating_2wk_std                      | 4주차 11위 (+0.231) → 6주차 14위 (0.166) ↓ | ⚠️ 약화
      └─ 4주차 11위 → 6주차 변화 추적
  rating_drift                        | 4주차  4위 (−0.363) → 6주차  4위 (0.528) = | ✅ 강하게 검증
      └─ 4주차 4위 일관 여부

📊 텍스트 피처 5개 (5주차 신규)
  evangelist_early_ratio              | SHAP rank  8 | 0.372  ★ TOP 10
      └─ ★ Q3 핵심 — fast_hit 초기 자발적 몰림
  topic_shift                         | SHAP rank 10 | 0.282  ★ TOP 10
      └─ fast_hit late−early = −10.8%p
  ad_ratio_2wk                        | SHAP rank 24 | 0.000  
      └─ 광고형 = 느린 hit 신호 후보
  voluntary_ratio_2wk                 | SHAP rank 20 | 0.017  
      └─ 보조 (전체 자발적 비율)
  topic_diversity                     | SHAP rank  6 | 0.397  ★ TOP 10
      └─ 보조 (Shannon entropy)

📊 카테고리 더미 무력화 (4주차 vs 6주차)
  sub_크림                    | 4주차  1위 (+0.422) → 6주차 16위 (0.027)
  sub_아이메이크업 

---
## Section 6. 에러 분석 — 4주차 FN 4개 추적

4주차에 놓친 FN 4개 (릴리바이레드·쏘내추럴 베이스, 이니스프리·프리메라 에센스):
- 공통 패턴: 평점 5.0, 분산 0, drift 0 → "느린 hit" 가설
- **6주차 C 모델이 이 4개를 잡는지 확인** = evangelist 2단계 가설의 실증


In [ ]:
# Section 6 — 4주차 FN 4개 추적 (6주차 C 모델이 잡았나?)
# 메모리 FN 4개: 릴리바이레드·쏘내추럴 베이스, 이니스프리·프리메라 에센스

# 1) DuckDB로 products에서 이름 부분 매칭 → product_id 추출
FN_NAME_PATTERNS = ['릴리바이레드', '쏘내추럴', '이니스프리', '프리메라']

# con은 셀 2에서 ATTACH 되어있음
fn_query = '''
SELECT product_id, name, brand, category_main, category_sub
FROM oy.products
WHERE name LIKE '%릴리바이레드%' OR brand LIKE '%릴리바이레드%'
   OR name LIKE '%쏘내추럴%'   OR brand LIKE '%쏘내추럴%'
   OR name LIKE '%이니스프리%' OR brand LIKE '%이니스프리%'
   OR name LIKE '%프리메라%'   OR brand LIKE '%프리메라%'
ORDER BY brand, name
'''
candidates = con.execute(fn_query).df()
print(f"이름 매칭 후보: {len(candidates)}개")
print(candidates[['product_id', 'name', 'brand', 'category_sub']].to_string())


이름 매칭 후보: 8개
      product_id                                                             name   brand category_sub
0  A000000224871                           [4월 올영픽/단독기획] 릴리바이레드 듀이 핏 팔레트 7 colors  릴리바이레드       아이메이크업
1  A000000137964                         [4월 올영픽/단독기획] 릴리바이레드 러브빔 치크밤 16종 (단품/기획)  릴리바이레드      베이스메이크업
2  A000000251137        [4월 올영픽/코코초 공동개발] 릴리바이레드 앙큼 라이어 코팅 틴트 (26AD) 3.6g (단품/기획)  릴리바이레드        립메이크업
3  A000000162114  [4월 올영픽 한정기획] 쏘내추럴 올 데이 타이트 메이크업 세팅 픽서 120ml 더블기획 (120ml+120ml)    쏘내추럴      베이스메이크업
4  A000000226481    [4월올영픽/120ml 대용량 출시/모공 블러] 쏘내추럴 올 데이 블러 메이크업 세팅 픽서 75ml/120ml    쏘내추럴      베이스메이크업
5  A000000138762                                   쏘내추럴 올 데이 타이트 메이크업 세팅 픽서 120ml    쏘내추럴      베이스메이크업
6  A000000230208                       [1+1/깐달걀피부결] 이니스프리 레티놀 시카 흔적 앰플 30ml 더블 기획   이니스프리    에센스/세럼/앰플
7  A000000231293                         [30g+15g] 프리메라 비타티놀 바운시 리프트 세럼 30g 기획/단품    프리메라    에센스/세럼/앰플


In [ ]:
# 6.2 4주차 model_A의 *진짜 FN* 추출 (메모리 의존 X, 데이터로 확인)
# FN = 실제 hit인데 model_A가 non-hit으로 예측한 상품

y_pred_A = model_A.predict(X_A_test_s)
fn_mask_in_test = (y_test == 1) & (y_pred_A == 0)
fn_count = fn_mask_in_test.sum()
print(f"4주차 model_A 실제 FN: {fn_count}개  (메모리에 4개로 기록)")

# FN의 features 행 추출 (X_A_test_s의 인덱스 = features의 원본 인덱스)
fn_indices = X_A_test_s.index[fn_mask_in_test]
fn_features = features.loc[fn_indices]

# product name 붙이기
fn_pids = fn_features['product_id'].tolist()
names_df = con.execute(
    f"SELECT product_id, name, brand, category_sub FROM oy.products WHERE product_id IN {tuple(fn_pids)}"
).df()

display_df = fn_features[['product_id', 'rating_2wk_mean', 'rating_2wk_std',
                          'rating_drift', 'reviews_2wk_count']].merge(
    names_df, on='product_id', how='left'
)
print()
print("=" * 100)
print("🎯 4주차 model_A의 진짜 FN — '느린 hit' 후보들")
print("=" * 100)
print(display_df.to_string(index=False))

# FN_IDS 자동 확정
FN_IDS = fn_pids
print(f"\n✅ FN_IDS 자동 확정: {len(FN_IDS)}개")


4주차 model_A 실제 FN: 4개  (메모리에 4개로 기록)

🎯 4주차 model_A의 진짜 FN — '느린 hit' 후보들
   product_id  rating_2wk_mean  rating_2wk_std  rating_drift  reviews_2wk_count                                                            name  brand category_sub
A000000137964         5.000000        0.000000      0.000000                  9                        [4월 올영픽/단독기획] 릴리바이레드 러브빔 치크밤 16종 (단품/기획) 릴리바이레드      베이스메이크업
A000000162114         5.000000        0.000000      0.000000                  4 [4월 올영픽 한정기획] 쏘내추럴 올 데이 타이트 메이크업 세팅 픽서 120ml 더블기획 (120ml+120ml)   쏘내추럴      베이스메이크업
A000000230208         4.954545        0.213201      0.076923                 22                      [1+1/깐달걀피부결] 이니스프리 레티놀 시카 흔적 앰플 30ml 더블 기획  이니스프리    에센스/세럼/앰플
A000000231293         4.652174        0.884652      0.492424                 23                        [30g+15g] 프리메라 비타티놀 바운시 리프트 세럼 30g 기획/단품   프리메라    에센스/세럼/앰플

✅ FN_IDS 자동 확정: 4개


In [ ]:
# 6.3 FN들의 6주차 C 모델 예측 + SHAP 비교
# FN_IDS는 셀 38에서 진짜 FN으로 자동 채워짐

# X_C에서 같은 인덱스 추출
fn_idx = fn_features.index
X_C_fn = X_C.loc[fn_idx]

# C 모델 예측 (hit 확률)
proba_fn = model_C.predict_proba(X_C_fn)[:, 1]

# SHAP 값 (FN에 대해 새로 계산)
shap_fn = explainer_C.shap_values(X_C_fn)
if isinstance(shap_fn, list):
    shap_fn = shap_fn[1]

# 결과 표 — 4주차 proba도 같이 보여주기
proba_A_fn = model_A.predict_proba(X_A_test_s.loc[fn_idx])[:, 1]

result = pd.DataFrame({
    'product_id': fn_features['product_id'].values,
    'is_hit_실제': fn_features['is_hit'].values,
    '4주차_proba': proba_A_fn,
    '6주차_proba': proba_fn,
    'Δ (6주-4주)': proba_fn - proba_A_fn,
    '6주차_잡혔나': ['✅ 잡힘' if p > 0.5 else '❌ 못 잡음' for p in proba_fn],
})
result = result.merge(names_df[['product_id', 'name']], on='product_id', how='left')[
    ['name', 'product_id', 'is_hit_실제', '4주차_proba', '6주차_proba', 'Δ (6주-4주)', '6주차_잡혔나']
]
print("=" * 100)
print("🎯 FN 추적 결과 — 4주차 vs 6주차 C 모델 비교")
print("=" * 100)
print(result.to_string(index=False))

caught = (proba_fn > 0.5).sum()
total = len(proba_fn)
print(f"\n→ {total}개 중 {caught}개 잡음 ({caught/total*100:.0f}%)")
if caught >= total // 2:
    print(f"   evangelist 2단계 가설 ✅ 부분 입증")
else:
    print(f"   evangelist 2단계 가설 ⚠️ 한계 노출 — 텍스트 피처도 못 잡는 케이스 존재")


🎯 FN 추적 결과 — 4주차 vs 6주차 C 모델 비교
                                                           name    product_id  is_hit_실제  4주차_proba  6주차_proba  Δ (6주-4주) 6주차_잡혔나
                       [4월 올영픽/단독기획] 릴리바이레드 러브빔 치크밤 16종 (단품/기획) A000000137964          1   0.229703   0.439729   0.210026  ❌ 못 잡음
[4월 올영픽 한정기획] 쏘내추럴 올 데이 타이트 메이크업 세팅 픽서 120ml 더블기획 (120ml+120ml) A000000162114          1   0.348410   0.728926   0.380517    ✅ 잡힘
                     [1+1/깐달걀피부결] 이니스프리 레티놀 시카 흔적 앰플 30ml 더블 기획 A000000230208          1   0.480112   0.764483   0.284371    ✅ 잡힘
                       [30g+15g] 프리메라 비타티놀 바운시 리프트 세럼 30g 기획/단품 A000000231293          1   0.491175   0.288162  -0.203014  ❌ 못 잡음

→ 4개 중 2개 잡음 (50%)
   evangelist 2단계 가설 ✅ 부분 입증


/Users/gimgyumin/Desktop/Developer/데이터 분석/oliveyoung-hit-prediction/.venv/lib/python3.12/site-packages/shap/explainers/_tree.py:620: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(


In [ ]:
# 6.4 SHAP waterfall plot — '어떤 피처가 이 상품을 hit/non-hit으로 미는가'
# 이모지(✅❌)는 한글 폰트에 없어서 경고 나오지만 PNG는 정상 저장

import shap as shap_module

# base value (모델의 평균 예측, hit 클래스 기준)
expected_value = explainer_C.expected_value
if isinstance(expected_value, (list, np.ndarray)) and len(np.atleast_1d(expected_value)) > 1:
    expected_value = expected_value[1]
else:
    expected_value = float(np.atleast_1d(expected_value)[0])

for i, (pid, name, proba) in enumerate(zip(
    result['product_id'].values, result['name'].values, result['6주차_proba'].values
)):
    short_name = name[:30] if len(name) > 30 else name
    catch = '[잡힘]' if proba > 0.5 else '[못잡음]'  # 이모지 대신 텍스트로
    
    expl = shap_module.Explanation(
        values=shap_fn[i],
        base_values=expected_value,
        data=X_C_fn.iloc[i].values,
        feature_names=X_C_fn.columns.tolist(),
    )
    
    plt.figure(figsize=(9, 6))
    shap_module.plots.waterfall(expl, max_display=12, show=False)
    plt.title(f"{catch} {short_name} - 6주차 proba {proba:.3f}")
    plt.tight_layout()
    safe_pid = pid.replace('/', '_')
    plt.savefig(f"../reports/figures/shap_waterfall_fn_{safe_pid}.png", dpi=120, bbox_inches='tight')
    plt.close()
    print(f"OK: shap_waterfall_fn_{safe_pid}.png ({short_name})")

print(f"\n→ reports/figures/ 폴더에서 waterfall {len(result)}장 확인")


/Users/gimgyumin/Desktop/Developer/데이터 분석/oliveyoung-hit-prediction/.venv/lib/python3.12/site-packages/shap/plots/_waterfall.py:279: UserWarning: Glyph 8722 (\N{MINUS SIGN}) missing from font(s) Apple SD Gothic Neo.
  text_bbox = txt_obj.get_window_extent(renderer=renderer)
/var/folders/65/xgrfhqb5421c768r34jn92f40000gn/T/ipykernel_87829/1059504603.py:31: UserWarning: Glyph 8722 (\N{MINUS SIGN}) missing from font(s) Apple SD Gothic Neo.
  plt.savefig(f"../reports/figures/shap_waterfall_fn_{safe_pid}.png", dpi=120, bbox_inches='tight')
/Users/gimgyumin/Desktop/Developer/데이터 분석/oliveyoung-hit-prediction/.venv/lib/python3.12/site-packages/shap/plots/_waterfall.py:279: UserWarning: Glyph 8722 (\N{MINUS SIGN}) missing from font(s) Apple SD Gothic Neo.
  text_bbox = txt_obj.get_window_extent(renderer=renderer)
/var/folders/65/xgrfhqb5421c768r34jn92f40000gn/T/ipykernel_87829/1059504603.py:29: UserWarning: Glyph 8722 (\N{MINUS SIGN}) missing from font(s) Apple SD Gothic Neo.
  plt.tight_layout

OK: shap_waterfall_fn_A000000137964.png ([4월 올영픽/단독기획] 릴리바이레드 러브빔 치크밤 1)
OK: shap_waterfall_fn_A000000162114.png ([4월 올영픽 한정기획] 쏘내추럴 올 데이 타이트 메이)


/Users/gimgyumin/Desktop/Developer/데이터 분석/oliveyoung-hit-prediction/.venv/lib/python3.12/site-packages/shap/plots/_waterfall.py:279: UserWarning: Glyph 8722 (\N{MINUS SIGN}) missing from font(s) Apple SD Gothic Neo.
  text_bbox = txt_obj.get_window_extent(renderer=renderer)
/var/folders/65/xgrfhqb5421c768r34jn92f40000gn/T/ipykernel_87829/1059504603.py:31: UserWarning: Glyph 8722 (\N{MINUS SIGN}) missing from font(s) Apple SD Gothic Neo.
  plt.savefig(f"../reports/figures/shap_waterfall_fn_{safe_pid}.png", dpi=120, bbox_inches='tight')
/Users/gimgyumin/Desktop/Developer/데이터 분석/oliveyoung-hit-prediction/.venv/lib/python3.12/site-packages/shap/plots/_waterfall.py:279: UserWarning: Glyph 8722 (\N{MINUS SIGN}) missing from font(s) Apple SD Gothic Neo.
  text_bbox = txt_obj.get_window_extent(renderer=renderer)
/var/folders/65/xgrfhqb5421c768r34jn92f40000gn/T/ipykernel_87829/1059504603.py:29: UserWarning: Glyph 8722 (\N{MINUS SIGN}) missing from font(s) Apple SD Gothic Neo.
  plt.tight_layout

OK: shap_waterfall_fn_A000000230208.png ([1+1/깐달걀피부결] 이니스프리 레티놀 시카 흔적 앰)
OK: shap_waterfall_fn_A000000231293.png ([30g+15g] 프리메라 비타티놀 바운시 리프트 세럼)

→ reports/figures/ 폴더에서 waterfall 4장 확인


---
## Section 7. 회고 + 7주차 결정

### 6주차 결과 요약 ✅

**3-way 비교 AUC**:

| 모델 | AUC | AP | recall(hit) | 발견 |
|---|---|---|---|---|
| A 로지스틱(19) | 0.5417 | 0.5245 | 0.667 | 4주차 베이스라인 정확 재현 (sanity check ✅) |
| B LightGBM(19) | 0.5417 | 0.5037 | 0.500 | **모델 변경만으론 AUC 0 변화** — 분산 가설은 SHAP에서만 살아남 |
| C LightGBM(24) | **0.6042** | 0.5440 | 0.417 | **텍스트 5개 추가로 +0.0625** ★ |

**핵심 발견 5가지**:

1. **분산 가설 정밀화** — slope 13위→2위 (살아남), drift 4위→4위 (일관), std 11위→14위 (약화). *"9피처가 균등하지 않다"* 5주차 가설이 모델 변별력으로 확인. 분산 가설은 *부서지지 않았고 정밀화*됨.
2. **카테고리 더미 무력화** — `sub_크림` 4주차 SHAP 1위(+0.422) → 6주차 16위(~0.027). `is_makeup` 2위→18위. **LightGBM이 4주차 FP 6개의 함정(카테고리 더미 과대평가)을 모델 안에서 알아서 회피**.
3. **텍스트 피처 TOP 10에 3개** — `topic_diversity` 6위, `evangelist_early_ratio` 8위, `topic_shift` 10위. `ad_ratio_2wk`는 24위(꼴찌, SHAP=0) — 5주차 광고형 분리 부족이 모델 변별력으로도 확인 (정직한 실패).
4. **evangelist_early_ratio 반전 발견** — dependence plot에서 비율 0.3+부터 SHAP −0.7~−1.0. *"evangelist 비율이 너무 높으면 *오히려* non-hit"*. 5주차 Q3 *"evangelist→대중 전환이 hit, 지속이 non-hit"*의 모델적 증명. **숫자(4주차) → 텍스트(5주차) → 모델 변별력(6주차) — 3번 독립 검증된 가설**.
5. **drift × count 상호작용 (신규)** — `rating_drift` dependence에서 색깔(`reviews_2wk_count`)에 따라 SHAP 방향 갈림. drift 음수 단독이 아니라 *count 적은 상품*에서만 hit 신호. 단변량 분석으론 절대 못 본 비선형 상호작용.

**4주차 FN 4개 추적 결과**:

| 상품 | 4주차 proba | 6주차 proba | Δ | 결과 |
|---|---|---|---|---|
| 쏘내추럴 픽서 | 0.348 | 0.729 | **+0.381** | ✅ 잡힘 |
| 이니스프리 레티놀 시카 앰플 | 0.480 | 0.764 | +0.284 | ✅ 잡힘 |
| 릴리바이레드 러브빔 치크밤 | 0.230 | 0.440 | +0.210 | ❌ 방향 ✓ 임계값 ✗ |
| 프리메라 비타티놀 세럼 | 0.491 | 0.288 | **−0.203** | ❌ **오히려 멀어짐** ★ |

평균 Δ = **+0.168 (양수)**. 2/4 잡음.

**메모리 정정 — 느린 hit 2가지 하위 패턴**:
- *조용한 만점형*: 평점 5.0/std 0/drift 0 (릴리바이레드·쏘내추럴·이니스프리)
- *분산형*: std 0.88/drift +0.49 (프리메라)

**★ 프리메라 케이스 (텍스트 피처 한계)**: 4주차에선 *간발의 차*로 못 잡았는데(0.491) 6주차엔 **−0.20 멀어짐**. dependence plot에서 *"std 큼 + drift 양수"* = 광고형 패턴으로 모델이 분류. **모델 개선이 모든 케이스에 단조 향상되지 않는다는 정직한 한계**.

---

### 7주차 방향 (CLAUDE.md 로드맵)
- **결과 정리 & 시각화** — `reports/figures/`에 6주차 발견 5가지를 차트로 압축. SHAP summary·dependence·waterfall PNG 9장 이미 확보
- **느린 hit 하위 유형 분리** 분석 (7주차 가설 정밀화 거리, 프리메라 케이스에서 도출)
- DuckDB 분석 쿼리 2~3개를 PostgreSQL 버전으로 작성 (포트폴리오 SQL 어필)
- 노션·블로그 6주차 결과 동기화

### 8주차 — 블로그 포스팅 + 발표 자료

---

### 면접 narrative 카드 (6주차 갱신, 총 5개)

1. **"같은 피처가 모델에 따라 정반대 신호"** — `velocity_slope` 4주차 13위(+0.097) → 6주차 SHAP 2위(0.64). 선형 모델이 평균만 봐서 못 잡던 *분산 신호*를 LightGBM이 분기로 분해. 4주차 narrative 데이터 검증.

2. **"분산 가설 정밀화 — 9피처가 균등하지 않다"** — slope/drift는 살아남, std는 약화. 5주차에 정직하게 적은 *유일한 예외*가 6주차 모델로 확인. *"분산 가설 부서졌다 / 살아있다"* 양극단이 아니라 *"피처별로 다르다"*는 진단.

3. **"카테고리 더미 무력화 — 모델이 함정을 알아서 회피"** — 4주차 로지스틱 FP 6개의 원인(`sub_크림` +0.422)이 LightGBM에서 SHAP ~0. *모델 자체가 4주차 한계를 진단하고 회피한* 증거. 모델 비교 narrative의 정점.

4. **"evangelist 가설 — 3번 독립 검증"** — 4주차 피처(숫자) → 5주차 텍스트 → 6주차 SHAP dependence. 같은 가설을 3가지 다른 데이터로 검증. *"숫자와 단어와 모델 변별력 모두 같은 그림"*. 가설의 robustness.

5. **"모델 개선의 정직한 한계 — 프리메라 케이스"** — 텍스트 피처 추가가 1개 케이스(분산형 느린 hit)는 *역효과*. **모든 개선이 모든 케이스에 단조 향상되지 않는다**는 분석가의 정직한 진단. 7주차 가설 정밀화 거리.

---

### 메모리 갱신 거리 (다음 세션 시작 시 참고)
- "느린 hit" = 2가지 하위 패턴 (조용한 만점형 / 분산형). 4주차 메모리는 *조용한 만점형*만 기록한 부분 정확
- FN 4개 product_id 확정: `A000000137964` (릴리바이레드 치크밤), `A000000162114` (쏘내추럴 픽서), `A000000230208` (이니스프리 앰플), `A000000231293` (프리메라 세럼)
- 6주차 평균 |SHAP| TOP 10: rating_2wk_mean(0.70) / velocity_slope(0.64) / discount_rate(0.61) / rating_drift(0.53) / review_length(0.43) / topic_diversity(0.40) / photo_review_ratio(0.38) / evangelist_early_ratio(0.37) / log_price(0.35) / topic_shift(0.28)
- 새 narrative 카드 5개 (위 1~5)
- 신규 발견: drift × count 상호작용, evangelist_early_ratio 반전 패턴
